In [1]:
from architecture import ConvBlock,Encoder,Encodings,CrossAttention,SelfAttention,FNN,BLOCK,Decoder,Model,Train

In [2]:
from get_batches import get_batches

In [3]:
import torch
import torch.nn as nn

In [4]:
model=Model(ConvBlock,Encoder,Encodings,CrossAttention,SelfAttention,FNN,BLOCK,Decoder)


In [5]:
decay=[]
no_decay=[]
for name,parameter in model.named_parameters():
    if parameter.ndim<2 or 'positional' in name:
        no_decay.append(parameter)
    else:
        decay.append(parameter)

In [6]:
optimizer=torch.optim.AdamW([{'params':decay,'weight_decay':1e-2},{'params':no_decay,'weight_decay':0.0}],lr=3e-4,betas=(0.9,0.999),eps=1e-8)

In [7]:
import math, torch

total_steps  = 50000
warmup_steps = 1000
base_lr      = 3e-4
min_lr       = 1e-6
min_ratio    = min_lr / base_lr          

def lr_lambda(step):
    if step < warmup_steps:                                   
        return 0.1 + 0.9 * step / warmup_steps                
    progress = (step - warmup_steps) / (total_steps - warmup_steps)  
    progress = min(progress, 1.0)                             
    cosine = 0.5 * (1 + math.cos(math.pi * progress))         
    return min_ratio + (1 - min_ratio) * cosine               


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

In [8]:
train=Train(model,optimizer,scheduler)

In [9]:
train.load('epoch24')

In [10]:
train.step

38400

In [ ]:
import csv
for k in range(1):
    with open('training_log.csv','a',newline='') as f:
        writer=csv.writer(f)
        for i in range(1600):
            lr=train.optimizer.param_groups[0]['lr']
            input,sar,target,mask=get_batches(train.step)
            input=input.to('cuda')
            sar=sar.to('cuda')
            target=target.to('cuda')
            mask=mask.to('cuda')
            loss,gn=train(input,sar,target,mask)
            writer.writerow([train.step,loss.item(),gn,lr])
            print(f'step-{train.step}, Loss={loss.item()}')

In [12]:
train.step

40000

In [13]:
train.save('epoch25')

In [14]:
p=next(train.model.parameters())
train.optimizer.state[p]['step']

tensor(40000.)

In [15]:
train.scheduler.last_epoch

40000

In [ ]:
import os
os.system("shutdown /s /t 0")